<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/chapter_07_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🐼 Chapter 7 — Pandas: Data Wrangling for the Real World</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Create, inspect, and filter Pandas DataFrames
- Handle missing values using drop, fill, and forward-fill strategies
- Add computed columns and use `pd.cut()` for binning
- Perform GroupBy aggregations and build pivot tables
- Merge DataFrames using join strategies (inner, left, outer)
- Build a complete EDA report function from scratch

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Pandas version:', pd.__version__)
print('✅ Setup complete!')

## 📋 Dataset Setup

Throughout this notebook we use a synthetic Indian employee dataset. Run this cell once and use `employees` in all exercises below.

### Data Dictionary

| Column | Type | Description |
|--------|------|-------------|
| `name` | str | Employee identifier |
| `department` | str | Engineering / Sales / HR / Finance |
| `experience` | int | Years of work experience (0–14) |
| `salary` | int | Annual salary in INR |
| `city` | str | Mumbai / Bangalore / Pune / Hyderabad / Delhi |
| `rating` | float | Performance rating 1–5 (some values missing) |
| `join_year` | int | Year the employee joined (2010–2023) |

In [ ]:
# ─────────────────────────────────────────────────────────────
# Dataset Generation — run this once
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
n = 200
employees = pd.DataFrame({
    'name':       ['Emp_' + str(i) for i in range(n)],
    'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Finance'], n),
    'experience': np.random.randint(0, 15, n),
    'salary':     np.random.randint(300000, 2000000, n),
    'city':       np.random.choice(['Mumbai', 'Bangalore', 'Pune', 'Hyderabad', 'Delhi'], n),
    'rating':     np.random.choice([1, 2, 3, 4, 5, np.nan], n),
    'join_year':  np.random.randint(2010, 2024, n),
})

print('Dataset shape:', employees.shape)
print('\nFirst 5 rows:')
employees.head()

## 📖 Section A — DataFrame and Series

A DataFrame is a table. Each column is a **Series** — a 1D labelled array.

```python
df.head()       # first 5 rows
df.shape        # (rows, columns)
df.dtypes       # column data types
df.info()       # dtypes + non-null counts
df.describe()   # statistics for numeric columns
```

> 💡 **Key Rule**
> `df.info()` is always your first call on a new dataset — it shows you missing values, types, and memory usage in one glance.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Inspecting a DataFrame
# ─────────────────────────────────────────────────────────────
print('Shape:', employees.shape)
print('\nColumn dtypes:')
print(employees.dtypes)
print('\nDescribe (numeric):')
print(employees.describe())
print('\nMissing values per column:')
print(employees.isnull().sum())

## 📖 Section B — Selecting Data

```python
df['col']                # → Series
df[['col1','col2']]      # → DataFrame (double brackets!)
df[df['col'] > value]    # row filter
df[(df['a'] > 1) & (df['b'] < 10)]  # compound — use & not 'and'
df.loc[row_label, col_label]    # label-based
df.iloc[row_int, col_int]       # position-based (like NumPy)
```

> 💡 **Key Rule**
> `.loc` is for labels — use column names and index values.
> `.iloc` is for integers — use 0-based positions, just like NumPy slicing.
> **Never mix them up.** `.loc[0, 'name']` ≠ `.iloc[0, 0]` if the index is not 0-based.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Selecting and Filtering
# ─────────────────────────────────────────────────────────────
print('Single column (Series):')
print(employees['department'].value_counts())

print('\nFiltered: Engineers earning > 1M:')
print(employees[(employees['department'] == 'Engineering') &
                (employees['salary'] > 1000000)][['name','salary','city']].head())

print('\n.iloc — rows 0-2, cols 1-3:')
print(employees.iloc[0:3, 1:4])

print('\n.loc — row 5, specific columns:')
print(employees.loc[5, ['name', 'department', 'salary']])

## 📖 Section C — Handling Missing Values

```python
df.isnull().sum()                    # count missing per column
df.isnull().sum() / len(df)          # fraction missing
df.dropna(subset=['col'])            # drop rows where col is NaN
df['col'].fillna(df['col'].median()) # fill with median
df['col'].fillna(df['col'].mode()[0])# fill with mode (most frequent)
df['col'].fillna(method='ffill')     # forward fill (time series)
```

> 💡 **Key Rule**
> If > 30% of a column is missing: **consider dropping the column**.
> For < 30%: fill with median (numeric) or mode (categorical).
> Never fill with mean if the column has outliers — the median is more robust.

## 📖 Section E — GroupBy and Aggregation

```python
df.groupby('col')['value'].mean()           # simple groupby
df.groupby('col').agg(                       # named agg
    avg_salary=('salary', 'mean'),
    count=('name', 'count')
)
df.pivot_table(values='salary', index='dept', columns='city', aggfunc='mean')
```

> 💡 **Key Rule**
> `.groupby()` alone returns a `GroupBy` object — it hasn't computed anything yet.
> You must chain `.mean()`, `.sum()`, `.agg()` etc. to trigger the computation.
> Think of it as: `groupby()` splits, aggregation function applies and combines.

---
## 🟢 Exercises — Level 1
---

> 🎯 **Exercise 7.1 — Inspect and Summarise** *(Level 1 · Est. 8 min)*
>
> a) Print shape, column names, dtypes; b) Show first 5 rows; c) Describe only numeric columns;
> d) Count missing values per column; e) What % of 'rating' is missing?

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.1: Inspect and Summarise
# Estimated time: 8 minutes
# ─────────────────────────────────────────────────────────────

# a) Shape, column names, dtypes
# YOUR CODE HERE

# b) First 5 rows
# YOUR CODE HERE

# c) Describe only numeric columns
# YOUR CODE HERE

# d) Missing values per column
missing = # YOUR CODE HERE
print('Missing values:\n', missing)

# e) Percentage of 'rating' that is missing
rating_missing_pct = # YOUR CODE HERE
print(f'\nRating missing: {rating_missing_pct:.1%}')

assert employees.shape == (200, 7), 'Dataset should be 200 rows × 7 columns'
assert 'rating' in missing.index,  'Missing values should include rating'
print('\n✅ Inspection complete!')

In [ ]:
# @title ✅ Solution — Exercise 7.1 (click to expand)
# @markdown > Run this cell to reveal the solution

# a) Shape, column names, dtypes
print('Shape:', employees.shape)
print('Columns:', employees.columns.tolist())
print('Dtypes:\n', employees.dtypes)

# b) First 5 rows
print('\nFirst 5 rows:')
print(employees.head())

# c) Describe numeric columns only — select_dtypes filters by type
print('\nNumeric summary:')
print(employees.select_dtypes(include='number').describe())

# d) Count nulls per column
missing = employees.isnull().sum()
print('\nMissing values:\n', missing[missing > 0])

# e) Percentage missing for rating
rating_missing_pct = employees['rating'].isnull().sum() / len(employees)
print(f'\nRating missing: {rating_missing_pct:.1%}')

assert employees.shape == (200, 7)
assert 'rating' in missing.index
print('\n✅ Solution verified!')

> 🎯 **Exercise 7.2 — Filtering and Selection** *(Level 1 · Est. 8 min)*
>
> a) All engineers earning > ₹1M; b) Employees in Bangalore OR Pune with experience > 5;
> c) Top 10 earners (name, dept, salary only); d) .loc to select salary > 1.5M → return name, city, salary

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.2: Filtering and Selection
# Estimated time: 8 minutes
# ─────────────────────────────────────────────────────────────

# a) Engineers earning > 1,000,000
engineers_1m = # YOUR CODE HERE
print(f'a) High-earning engineers: {len(engineers_1m)}')

# b) (Bangalore OR Pune) AND experience > 5
bg_or_pune_exp5 = # YOUR CODE HERE
print(f'b) Bangalore/Pune with 5+ exp: {len(bg_or_pune_exp5)}')

# c) Top 10 earners — name, department, salary
top_10 = # YOUR CODE HERE
print('\nc) Top 10 earners:')
print(top_10)

# d) .loc — salary > 1.5M, return only name, city, salary
high_earners = # YOUR CODE HERE
print(f'\nd) Employees earning > 1.5M: {len(high_earners)}')

assert 'name' in top_10.columns and 'salary' in top_10.columns
assert len(top_10) == 10
assert set(high_earners.columns) == {'name', 'city', 'salary'}
print('\n✅ Filtering exercises verified!')

In [ ]:
# @title ✅ Solution — Exercise 7.2 (click to expand)
# @markdown > Run this cell to reveal the solution

# a) Compound filter — both conditions must be true (&)
engineers_1m = employees[(employees['department'] == 'Engineering') &
                          (employees['salary'] > 1000000)]
print(f'a) High-earning engineers: {len(engineers_1m)}')

# b) OR conditions use | — parentheses required around each condition
bg_or_pune_exp5 = employees[
    ((employees['city'] == 'Bangalore') | (employees['city'] == 'Pune')) &
    (employees['experience'] > 5)
]
print(f'b) Bangalore/Pune with 5+ years: {len(bg_or_pune_exp5)}')

# c) sort_values descending, take first 10, select columns
top_10 = employees.sort_values('salary', ascending=False)[['name','department','salary']].head(10)
print('\nc) Top 10 earners:')
print(top_10.to_string(index=False))

# d) .loc with boolean mask and column list
mask = employees['salary'] > 1500000
high_earners = employees.loc[mask, ['name', 'city', 'salary']]
print(f'\nd) Employees earning > 1.5M: {len(high_earners)}')

assert len(top_10) == 10
assert set(high_earners.columns) == {'name', 'city', 'salary'}
print('\n✅ Solution verified!')

---
## 🟡 Exercises — Level 2
---

> 🎯 **Exercise 7.3 — Cleaning and Transforming** *(Level 2 · Est. 15 min)*
>
> a) Fill missing 'rating' with median; b) Add 'salary_lpa' = salary/100000 rounded to 2dp;
> c) Add 'seniority' using pd.cut on experience; d) Add 'tenure_years' = 2026 - join_year

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.3: Cleaning and Transforming
# Estimated time: 15 minutes
# ─────────────────────────────────────────────────────────────

# Work on a copy to preserve original
emp = employees.copy()

# a) Fill missing 'rating' with the column's median
# YOUR CODE HERE

# b) Add 'salary_lpa' = salary / 100000, rounded to 2 decimal places
# YOUR CODE HERE

# c) Add 'seniority' using pd.cut on experience
#    [0,2) = 'Junior', [2,5) = 'Mid', [5,10) = 'Senior', [10,15] = 'Principal'
# YOUR CODE HERE

# d) Add 'tenure_years' = 2026 - join_year
# YOUR CODE HERE

print(emp[['experience', 'seniority', 'salary', 'salary_lpa', 'rating', 'tenure_years']].head(8))

assert emp['rating'].isnull().sum() == 0,      'Rating should have no missing values after fill'
assert 'salary_lpa' in emp.columns,            'salary_lpa column must be added'
assert 'seniority'  in emp.columns,            'seniority column must be added'
assert 'tenure_years' in emp.columns,          'tenure_years column must be added'
assert emp['seniority'].isnull().sum() == 0,   'seniority should have no NaN'
print('\n✅ Data cleaning and transformation verified!')

In [ ]:
# @title ✅ Solution — Exercise 7.3 (click to expand)
# @markdown > Run this cell to reveal the solution

emp = employees.copy()

# a) Fill NaN with median — median is robust to outliers
emp['rating'] = emp['rating'].fillna(emp['rating'].median())

# b) Create new computed column with round()
emp['salary_lpa'] = (emp['salary'] / 100000).round(2)

# c) pd.cut bins a continuous variable into labelled categories
#    right=False → left-inclusive intervals [a, b)
emp['seniority'] = pd.cut(
    emp['experience'],
    bins=[0, 2, 5, 10, 15],
    labels=['Junior', 'Mid', 'Senior', 'Principal'],
    right=False,
    include_lowest=True
)

# d) Scalar subtraction is vectorised — no loop needed
emp['tenure_years'] = 2026 - emp['join_year']

print(emp[['experience', 'seniority', 'salary', 'salary_lpa', 'rating', 'tenure_years']].head(8))

assert emp['rating'].isnull().sum() == 0
assert 'salary_lpa'   in emp.columns
assert 'seniority'    in emp.columns
assert 'tenure_years' in emp.columns
print('\n✅ Solution verified! pd.cut() is one of the most-used feature engineering tools in real ML pipelines.')

> 🎯 **Exercise 7.4 — GroupBy Analysis** *(Level 2 · Est. 15 min)*
>
> a) Average salary per department (sorted highest first); b) Per city: avg salary, max experience, count (use .agg());
> c) Which seniority level has the best avg rating?; d) Pivot table: departments as rows, cities as columns, mean salary as values

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.4: GroupBy Analysis
# Estimated time: 15 minutes
# ─────────────────────────────────────────────────────────────

# Use emp from previous exercise (already has seniority column)
# If not available, run ex73-solution first

# a) Average salary per department, sorted highest first
dept_salary = # YOUR CODE HERE
print('a) Avg salary by department:')
print(dept_salary)

# b) Per city: avg salary, max experience, count of employees
city_summary = # YOUR CODE HERE
print('\nb) City summary:')
print(city_summary)

# c) Which seniority level has best average rating?
seniority_ratings = # YOUR CODE HERE
best_seniority = # YOUR CODE HERE
print(f'\nc) Best average rating by seniority: {best_seniority}')

# d) Pivot table: rows=department, cols=city, values=mean salary
pivot = # YOUR CODE HERE
print('\nd) Salary pivot table (department × city):')
print(pivot.round(0))

assert isinstance(dept_salary, pd.Series), 'dept_salary should be a Series'
assert pivot.shape[0] == 4, 'Pivot should have 4 department rows'
print('\n✅ GroupBy exercises verified!')

In [ ]:
# @title ✅ Solution — Exercise 7.4 (click to expand)
# @markdown > Run this cell to reveal the solution

# Rebuild emp if needed
emp = employees.copy()
emp['rating'] = emp['rating'].fillna(emp['rating'].median())
emp['salary_lpa'] = (emp['salary'] / 100000).round(2)
emp['seniority'] = pd.cut(emp['experience'], bins=[0,2,5,10,15],
                           labels=['Junior','Mid','Senior','Principal'],
                           right=False, include_lowest=True)
emp['tenure_years'] = 2026 - emp['join_year']

# a) groupby → mean → sort_values
dept_salary = emp.groupby('department')['salary'].mean().sort_values(ascending=False)
print('a) Avg salary by department:')
print(dept_salary.apply(lambda x: f'₹{x:,.0f}'))

# b) .agg() with named aggregations — clean and readable
city_summary = emp.groupby('city').agg(
    avg_salary  = ('salary',     'mean'),
    max_exp     = ('experience', 'max'),
    emp_count   = ('name',       'count')
).sort_values('avg_salary', ascending=False)
print('\nb) City summary:')
print(city_summary.round(0))

# c) GroupBy seniority, mean rating, then idxmax for top entry
seniority_ratings = emp.groupby('seniority')['rating'].mean().sort_values(ascending=False)
best_seniority    = seniority_ratings.idxmax()
print(f'\nc) Best seniority by rating: {best_seniority}  ({seniority_ratings[best_seniority]:.2f})')

# d) pivot_table — same as an Excel PivotTable
pivot = pd.pivot_table(emp, values='salary', index='department', columns='city', aggfunc='mean')
print('\nd) Pivot table:')
print(pivot.round(0))

assert isinstance(dept_salary, pd.Series)
assert pivot.shape[0] == 4
print('\n✅ Solution verified! Pivot tables are Excel cross-tabs in Python — same mental model, more power.')

---
## 🔴 Exercises — Level 3
---

> 🎯 **Exercise 7.5 — Full EDA Pipeline** *(Level 3 · Est. 25 min)*
>
> Write `eda_report(df)` returning a dict with: shape, missing_pct, numeric_summary, top_earner, dept_summary, salary_distribution (Q1, median, Q3, IQR).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.5: Full EDA Pipeline
# Estimated time: 25 minutes
# ─────────────────────────────────────────────────────────────

def eda_report(df):
    """
    Generate a comprehensive EDA report for a DataFrame.
    Returns a dict with shape, missing_pct, numeric_summary,
    top_earner, dept_summary, and salary_distribution.
    """
    report = {}
    # YOUR CODE HERE
    return report

report = eda_report(employees)
print('EDA Report Keys:', list(report.keys()))
print('\nShape:', report.get('shape'))
print('\nTop Earner:', report.get('top_earner'))
print('\nSalary Distribution:', report.get('salary_distribution'))

assert 'shape'               in report, 'Missing: shape'
assert 'missing_pct'         in report, 'Missing: missing_pct'
assert 'top_earner'          in report, 'Missing: top_earner'
assert 'salary_distribution' in report, 'Missing: salary_distribution'
assert 'IQR'                 in report['salary_distribution'], 'salary_distribution must include IQR'
print('\n✅ EDA pipeline complete!')

In [ ]:
# @title ✅ Solution — Exercise 7.5 (click to expand)
# @markdown > Run this cell to reveal the solution

def eda_report(df):
    """Comprehensive EDA report for any DataFrame."""

    # Shape
    shape = df.shape

    # Missing percentage per column
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

    # Numeric summary
    numeric_summary = df.select_dtypes(include='number').describe()

    # Top earner — row with max salary as a dict
    top_row = df.loc[df['salary'].idxmax()]
    top_earner = {
        'name':       top_row['name'],
        'salary':     top_row['salary'],
        'department': top_row['department'],
        'city':       top_row['city'],
    }

    # Department summary
    dept_summary = df.groupby('department').agg(
        avg_salary = ('salary', 'mean'),
        count      = ('name',   'count')
    ).round(0)

    # Salary distribution
    q1     = df['salary'].quantile(0.25)
    median = df['salary'].median()
    q3     = df['salary'].quantile(0.75)
    salary_dist = {'Q1': q1, 'median': median, 'Q3': q3, 'IQR': q3 - q1}

    return {
        'shape':               shape,
        'missing_pct':         missing_pct,
        'numeric_summary':     numeric_summary,
        'top_earner':          top_earner,
        'dept_summary':        dept_summary,
        'salary_distribution': salary_dist,
    }

report = eda_report(employees)
print('Shape:',           report['shape'])
print('Top Earner:',      report['top_earner'])
print('Salary Dist:',     report['salary_distribution'])
print('\nDept Summary:')
print(report['dept_summary'])

assert 'shape'               in report
assert 'IQR'                 in report['salary_distribution']
print('\n✅ Solution verified! This eda_report() pattern is used in every professional data science notebook.')

> 🎯 **Exercise 7.6 — Merge and Join** *(Level 3 · Est. 20 min)*
>
> Create a `benefits` DataFrame. Merge with employees on 'department'. Analyse total meal allowance cost per city and find city with most employees without health insurance.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 7.6: Merge and Join
# Estimated time: 20 minutes
# ─────────────────────────────────────────────────────────────

benefits = pd.DataFrame({
    'department':      ['Engineering', 'Sales', 'HR', 'Finance'],
    'health_insurance': [True, True, False, True],
    'meal_allowance':  [5000, 3000, 2000, 4000],
})

# a) Merge employees with benefits on 'department' (inner join)
merged = # YOUR CODE HERE
print(f'a) Merged shape: {merged.shape}')
print(merged[['name','department','city','health_insurance','meal_allowance']].head())

# b) Total monthly meal allowance cost per city
meal_cost_by_city = # YOUR CODE HERE
print('\nb) Monthly meal allowance cost per city:')
print(meal_cost_by_city)

# c) Which city has the most employees WITHOUT health insurance?
no_insurance_city = # YOUR CODE HERE
print(f'\nc) City with most employees lacking insurance: {no_insurance_city}')

assert 'health_insurance' in merged.columns, 'merged must have health_insurance'
assert 'meal_allowance'   in merged.columns, 'merged must have meal_allowance'
assert merged.shape[0] == 200, 'Inner join should preserve all 200 employees (all depts match)'
print('\n✅ Merge and join exercises verified!')

In [ ]:
# @title ✅ Solution — Exercise 7.6 (click to expand)
# @markdown > Run this cell to reveal the solution

benefits = pd.DataFrame({
    'department':       ['Engineering', 'Sales', 'HR', 'Finance'],
    'health_insurance': [True, True, False, True],
    'meal_allowance':   [5000, 3000, 2000, 4000],
})

# a) Inner join — only rows with matching keys in both DataFrames
merged = pd.merge(employees, benefits, on='department', how='inner')
print(f'a) Merged shape: {merged.shape}')
print(merged[['name','department','city','health_insurance','meal_allowance']].head())

# b) GroupBy city → sum the meal_allowance (per employee × monthly rate)
meal_cost_by_city = merged.groupby('city')['meal_allowance'].sum().sort_values(ascending=False)
print('\nb) Monthly meal allowance cost per city (₹):')
print(meal_cost_by_city.apply(lambda x: f'₹{x:,}'))

# c) Filter no-insurance rows, count per city, find argmax
no_insurance = merged[merged['health_insurance'] == False]
no_insurance_city = no_insurance.groupby('city')['name'].count().idxmax()
print(f'\nc) City with most uninsured employees: {no_insurance_city}')

assert 'health_insurance' in merged.columns
assert merged.shape[0] == 200
print('\n✅ Solution verified! pd.merge() is SQL JOIN in Python — the same logic, just with DataFrames.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is the difference between .loc and .iloc?</strong></summary>

**Answer:** `.loc` is **label-based** — you access by column name and index label. `.iloc` is **position-based** — you access by integer position (0-indexed). When the DataFrame index is 0,1,2... they behave similarly. But if you've reset the index or have a custom string index, they differ. Always prefer `.loc` when you know the column names (clearer, safer) and `.iloc` when you need positional slicing (like NumPy).
</details>

<details>
<summary><strong>Q2: How do you handle missing values? When to drop vs fill?</strong></summary>

**Answer:** Drop when: > 30% of a column is missing (imputing that much introduces too much noise), or when the column is not important. Fill when: < 30% missing and the column is important. Fill strategy: numeric → median (robust to outliers), categorical → mode (most frequent value), time series → forward fill (ffill) to carry the last known value forward.
</details>

<details>
<summary><strong>Q3: What is the difference between merge() and concat()?</strong></summary>

**Answer:** `pd.merge()` combines DataFrames **horizontally** based on a matching key column — like a SQL JOIN. `pd.concat()` **stacks** DataFrames either vertically (rows) or horizontally (columns) by position/index — like SQL UNION ALL. Use merge() when you have a key relationship. Use concat() when you're stacking additional data of the same structure (e.g., monthly CSV files).
</details>

<details>
<summary><strong>Q4: Explain groupby — what does it return before aggregation?</strong></summary>

**Answer:** `df.groupby('col')` returns a `DataFrameGroupBy` object — a lazy container that has split the data into groups but hasn't done any computation yet. No result is computed until you call an aggregation function (`.mean()`, `.sum()`, `.agg()` etc.). This is the "split-apply-combine" pattern: split by group, apply a function, combine results into a new DataFrame.
</details>

<details>
<summary><strong>Q5: How would you identify and remove duplicate rows?</strong></summary>

**Answer:** `df.duplicated()` returns a boolean Series — True for duplicate rows. `df.duplicated().sum()` counts them. `df.drop_duplicates()` removes them, keeping the first occurrence by default. To check duplicates on specific columns: `df.duplicated(subset=['col1','col2'])`. Always check for near-duplicates too (same person, slightly different spelling) — those require fuzzy matching.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 7 Complete!</h3>
<p>Outstanding work! You've completed a real-world data wrangling workflow:</p>
<ul>
<li>Inspecting and filtering DataFrames with .loc / .iloc</li>
<li>Handling missing values intelligently (fill vs drop)</li>
<li>Adding computed columns and binning with pd.cut()</li>
<li>GroupBy aggregations with .agg() and pivot tables</li>
<li>Building a reusable EDA report function</li>
<li>Merging DataFrames — the SQL JOIN equivalent in Pandas</li>
</ul>
<p><strong>Bonus:</strong> End-to-End Mini ML Project — bringing Chapters 4–7 together | <a href='#'>Open Bonus Notebook →</a></p>
</div>